In [41]:
import pandas as pd
import utils as u
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mir_eval
import numpy as np
from chroma_transformer import KeyLabelConverter

# Load data

In [7]:
metadata_df = pd.read_csv("baseline_hpc/fmak_v2.csv")
actual_keys_dict = {}

for _, row in metadata_df.iterrows():
    actual_keys_dict[row['track_id']] = u.normalize_key(row['key_and_mode'])

print(f"{len(actual_keys_dict)} actual keys loaded.")

5489 actual keys loaded.


In [12]:
krumhansl_df = pd.read_csv("baseline_hpc/krumhansl_predictions_full.csv")
krumhansl_keys_dict = {}

for _, row in krumhansl_df.iterrows():
    krumhansl_keys_dict[row['track_id']] = u.normalize_key(row['predicted_key'])

print(f"{len(krumhansl_keys_dict)} Krumhansl predictions loaded.")

5489 Krumhansl predictions loaded.


In [13]:
madmom_df = pd.read_csv("baseline_hpc/madmom_predictions_full.csv")
madmom_keys_dict = {}

for _, row in madmom_df.iterrows():
    madmom_keys_dict[row['track_id']] = u.normalize_key(row['predicted_key'])

print(f"{len(madmom_keys_dict)} Madmom predictions loaded.")

5489 Madmom predictions loaded.


# Evaluate baseline methods on entire dataset

In [20]:
track_ids = list(actual_keys_dict.keys())
y_true = [actual_keys_dict[tid] for tid in track_ids]
y_pred_krumhansl = [krumhansl_keys_dict.get(tid) for tid in track_ids]
y_pred_madmom = [madmom_keys_dict.get(tid) for tid in track_ids]

def print_metrics(y_true, y_pred, name):
    print(f"=== {name} ===")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision (Micro):    {precision_score(y_true, y_pred, average='micro'):.4f}")
    print(f"Precision (Macro):    {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Precision (Weighted): {precision_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"Recall (Micro):       {recall_score(y_true, y_pred, average='micro'):.4f}")
    print(f"Recall (Macro):       {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Recall (Weighted):    {recall_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"F1 (Micro):           {f1_score(y_true, y_pred, average='micro'):.4f}")
    print(f"F1 (Macro):           {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1 (Weighted):        {f1_score(y_true, y_pred, average='weighted'):.4f}")
    print()

print_metrics(y_true, y_pred_krumhansl, "Krumhansl")
print_metrics(y_true, y_pred_madmom, "Madmom")

=== Krumhansl ===
Accuracy:  0.5360
Precision (Micro):    0.5360
Precision (Macro):    0.5098
Precision (Weighted): 0.5446
Recall (Micro):       0.5360
Recall (Macro):       0.5096
Recall (Weighted):    0.5360
F1 (Micro):           0.5360
F1 (Macro):           0.5062
F1 (Weighted):        0.5370

=== Madmom ===
Accuracy:  0.6533
Precision (Micro):    0.6533
Precision (Macro):    0.6472
Precision (Weighted): 0.6598
Recall (Micro):       0.6533
Recall (Macro):       0.6319
Recall (Weighted):    0.6533
F1 (Micro):           0.6533
F1 (Macro):           0.6351
F1 (Weighted):        0.6523



In [28]:
def get_avg_weighted_score(y_true, y_pred):
    cumulative_score = 0.0

    for true, pred in zip(y_true, y_pred):
        weighted_score = mir_eval.key.weighted_score(true, pred)
        cumulative_score += weighted_score

    avg_score = cumulative_score / len(y_true)
    return avg_score

avg_weighted_krumhansl = get_avg_weighted_score(y_true, y_pred_krumhansl)
avg_weighted_madmom = get_avg_weighted_score(y_true, y_pred_madmom)

print(f"Average Weighted Score (Krumhansl): {avg_weighted_krumhansl:.4f}")
print(f"Average Weighted Score (Madmom):    {avg_weighted_madmom:.4f}")

Average Weighted Score (Krumhansl): 0.6154
Average Weighted Score (Madmom):    0.7111


# For Chroma Transformer, aggregate across folds

In [30]:
def get_fold_ids(fold_num):
    fold_df = pd.read_csv(f"chroma_transformer/folds/fold_{fold_num:02d}/test.csv")
    return fold_df['track_id'].tolist()

fold_track_ids = [get_fold_ids(fold_num) for fold_num in range(10)]
print(f"Loaded {len(fold_track_ids)} fold track ID lists with lengths: {[len(ids) for ids in fold_track_ids]}")

Loaded 10 fold track ID lists with lengths: [549, 549, 549, 549, 549, 549, 549, 549, 549, 548]


In [38]:
np.load(f"chroma_transformer/predictions/fold_00_predictions.npz")

NpzFile 'chroma_transformer/predictions/fold_00_predictions.npz' with keys: y_true, y_pred, probs

In [45]:
def get_chroma_transformer_predictions(fold_num):
    import numpy as np
    pred_file = f"chroma_transformer/predictions/fold_{fold_num:02d}_predictions.npz"
    data = np.load(pred_file)
    predictions = data['y_pred']
    keys = [KeyLabelConverter.label_to_key(pred) for pred in predictions]
    normalized_keys = [u.normalize_key(key) for key in keys]
    fold_dict = {tid: norm_key for tid, norm_key in zip(fold_track_ids[fold_num], normalized_keys)}
    return fold_dict

def get_baseline_predictions(fold_num, baseline_keys_dict):
    track_ids = fold_track_ids[fold_num]
    pred_dict = {tid: baseline_keys_dict.get(tid) for tid in track_ids}
    return pred_dict

chroma_transformer_predictions = [get_chroma_transformer_predictions(fold_num) for fold_num in range(10)]
krumhansl_predictions = [get_baseline_predictions(fold_num, krumhansl_keys_dict) for fold_num in range(10)]
madmom_predictions = [get_baseline_predictions(fold_num, madmom_keys_dict) for fold_num in range(10)]
true_predictions = [get_baseline_predictions(fold_num, actual_keys_dict) for fold_num in range(10)]

In [56]:
def get_fold_metrics(fold_num):
    track_ids = fold_track_ids[fold_num]
    y_true = [true_predictions[fold_num][tid] for tid in track_ids]
    y_pred_chroma = [chroma_transformer_predictions[fold_num][tid] for tid in track_ids]
    print_metrics(y_true, y_pred_chroma, f"Chroma Transformer - Fold {fold_num}") 

get_fold_metrics(9)

=== Chroma Transformer - Fold 9 ===
Accuracy:  0.6679
Precision (Micro):    0.6679
Precision (Macro):    0.6310
Precision (Weighted): 0.6865
Recall (Micro):       0.6679
Recall (Macro):       0.6352
Recall (Weighted):    0.6679
F1 (Micro):           0.6679
F1 (Macro):           0.6237
F1 (Weighted):        0.6712

